In [ ]:
# ===== Standard library =====
import csv
import datetime
import glob
import os
import random
import re
from collections import Counter
from datetime import timedelta

# ===== Numerical & data handling =====
import numpy as np
import pandas as pd
import polars as pl
from numpy.random import default_rng

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ===== Scikit-learn =====
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    KFold,
    train_test_split,
)
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler,
)
from sklearn.utils.class_weight import compute_class_weight

# ===== Imbalanced-learn =====
from imblearn.under_sampling import RandomUnderSampler

# ===== TensorFlow / Keras =====
import tensorflow as tf
from keras.utils import to_categorical
from tcn import TCN
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from tensorflow.keras.layers import (
    Activation,
    Add,
    BatchNormalization,
    Concatenate,
    Conv1D,
    Dense,
    Dropout,
    Embedding,
    Flatten,
    GlobalAveragePooling1D,
    Input,
    MaxPooling1D,
)
from tensorflow.keras.models import Model, Sequential

# ===== Machine learning / optimization =====
import joblib
import lightgbm as lgb
import optuna
import shap

# ===== Project-specific modules =====
import utils.data_integration as data_integration
import utils.data_preprocessing as data_preprocessing

In [3]:
# convert all sublists to arrays
def flatten(subject):
    array_data = [np.array(x) for x in subject]  
    flattened_data = np.concatenate(array_data, axis=0)
    return flattened_data

## read and prepare external datasets

In [ ]:

# Path to the root folder
root_folder = "Data/AZT1D 2025/CGM Records"  

# List to hold each DataFrame
all_dfs = []
i = 1

# Walk through the directory
for dirpath, dirnames, filenames in os.walk(root_folder):
    for filename in filenames:
        if filename.endswith(".csv"):
            file_path = os.path.join(dirpath, filename)
            try:
                df = pd.read_csv(file_path)
                df["PtID"] = i 
                all_dfs.append(df)
                i = i + 1
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
# Concatenate all DataFrames into one
if all_dfs:
    df_AZT125 = pd.concat(all_dfs, ignore_index=True)
else:
    print("No CSV files found.")

df_AZT125["ts"] = pd.to_datetime(df_AZT125["EventDateTime"])
df_AZT125 = df_AZT125.rename(columns={"CGM": "GlucoseCGM"})
df_AZT125["PtID"] = df_AZT125["PtID"].astype(str) + "_AZT125"
# resamples to 5 minute intervals to have unifrom sample rate; this is done for each subject seperately
df_AZT125 = data_integration.df_resample(df_AZT125, timestamp = "ts", frequency= "5min", mode="glucose", fillid = "PtID", value = "GlucoseCGM")
# adds a "Database" column with the name of the Dataset to enable reidentification
# reads dataframe with demographic information

df_AZT125["Database"] = "AZT125"
df_AZT125 = df_AZT125[["ts", "GlucoseCGM",	"PtID",	"Database"]]


In [7]:
def remove_outliers(df_org, value, modus, subject = "PtID"):
    """
    This function is used to identify and remove outliers with the interquartile range method.
    Parameters:
   - df_org is the original dataset, 
   - value is the name of the column with the values of interest containing outliers,
   - the modus—either "glucose" or "vitals"—specifies the requirements for outlier removal,
   - subject is the subject IDfillid is the name of the column which needs fillforward to impute produced missing values.
    Output: It returns the database with removed outliers in the specified column.
   """ 
    data = df_org.copy()
    # repaces zero values with nan values
    #data[value].replace(0, np.nan, inplace=True) 
    data[value] = data[value].replace(0, np.nan)
    # computes group-wise statistics, identifying the first and third quartiles
    Q1 = data.groupby(subject)[value].transform(lambda x: x.quantile(0.25))
    Q3 = data.groupby(subject)[value].transform(lambda x: x.quantile(0.75))
    IQR = Q3 - Q1

    # computes lower and upper thresholds
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    if modus == "glucose":
      # identifies outliers which are not in the range of 25 and 75 percent of values and replaces them with nan values
      # also removes CGM values which are less than 40 mg/dL
      is_outlier = (data[value] < lower) | (data[value] > upper) | (data[value] < 40) | (data[value] > 500) 
    elif modus == "vitals":
      is_outlier = (data[value] < lower) | (data[value] > upper) | (data[value] < 30) 
    else:
      print("modus must be glucose or vitals")

    # outliers in the value column are replaced with nan values
    data.loc[is_outlier, value] = np.nan

    return data

In [8]:
cleaned_data = remove_outliers(df_AZT125, value= "GlucoseCGM", modus= "glucose", subject = "PtID") 
data_cleanedpd = cleaned_data.sort_values(by=["PtID", "ts"]).reset_index(drop=True)

# linear interpolation is used for gaps which are less than 30 minute (6 consecuitve datapoints)
data_cleanedpd["GlucoseCGM"] = data_cleanedpd.groupby("PtID")["GlucoseCGM"].transform(
    lambda x: data_preprocessing.gap_limited_interpolation(x, limit=6)
)

# stineman interpolation for values between 30 adn 120 minutes
df_interpolated_stine = data_cleanedpd.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.interpolate_stineman_group(x, timestamp = "ts", value = "GlucoseCGM", llimit=6, ulimit=24))
df_interpolated_stine = df_interpolated_stine.sort_index()

df_interpolated_stine = df_interpolated_stine[["ts", "GlucoseCGM", "PtID"]]
df_interpolated_stine

,ts,GlucoseCGM,PtID
0,2024-02-06 22:00:00,126.0,10_AZT125
1,2024-02-06 22:05:00,128.0,10_AZT125
2,2024-02-06 22:10:00,128.0,10_AZT125
3,2024-02-07 15:10:00,155.0,10_AZT125
4,2024-02-07 15:15:00,151.0,10_AZT125
...,...,...,...
35131,2024-02-29 23:40:00,139.0,9_AZT125
35132,2024-02-29 23:45:00,138.0,9_AZT125
35133,2024-02-29 23:50:00,138.0,9_AZT125
35134,2024-02-29 23:55:00,141.0,9_AZT125


In [9]:
import datetime 

# generates classes for the hypoglycemia classification task
def class_generation(df_copy: pd.DataFrame, timestamp_col: str, start: int, end: int, class_number: int, col_name: str) -> pl.DataFrame: 
    """
    This function is used to assign labels.
    Parameters:
   - df_copy is the original pandas dataframe, 
   - timestamp_col is the name of the column with the timestamps, 
   - start is time in minutes of the minimum duration before hypogylcemia
   - end is time in minutes of the maximum duration before hypogylcemia
   - class_number is the class for the defined time range before hypoglycemia
    Output: It returns the dataframe with the assigned class for the spefied time range before hypoglycemia.
   """ 
    # first, the dataframe is converted to polars to increase efficacy
    df = pl.from_pandas(df_copy)
    # timestamps are sorted 
    df = df.sort(timestamp_col)
    
    # zero events are hypoglycemic data points
    # these are selected and the timestamps are stores as a series
    event_times = df.filter(pl.col(col_name) == 0).select(timestamp_col).to_series()

    # the start and end time of the time range before hypogylcemia are computed
    start_bounds = event_times - datetime.timedelta(minutes=start)
    end_bounds = event_times - datetime.timedelta(minutes=end)

    # the series is masked 
    mask = pl.Series([False] * df.height)

    # timestamps meeting the defined criteria of specified time bonds are identified 
    # moreover, the class should be -1 (not assinged as other classes before)
    for start_time, end_time in zip(start_bounds, end_bounds):
        window_mask = (
            (df[timestamp_col] > end_time) &
            (df[timestamp_col] <= start_time) &
            (df[col_name] == -1)
        )
        mask = mask | window_mask

    df = df.with_columns([
        pl.when(mask).then(class_number).otherwise(pl.col(col_name)).alias(col_name)
    ])

    # the dataframe is converted back to pandas 
    df_pd = df.to_pandas()
    # the dataframe with assigned classes is returned
    return df_pd

#test_data_classes = []
#for i in test_data_cleaned:
classes_df = df_interpolated_stine.sort_values(by=["PtID", "ts"]).reset_index(drop=True)
                                                    
# initially all classes are assigned a class -1
classes_df["Class1"] = -1

# hypoglycemic values are defined as class 0 with values equal and lower than 70 mg/dL
classes_df.loc[classes_df["GlucoseCGM"] <= 70, "Class1"] = 0

# calls the function class_generation to assing classes based on wanted intervals before hypogylcemia for each subject separately
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 5, 20, 1, col_name = "Class1")) # 5-15 min
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 20, 50, 2, col_name = "Class1")) # 15-30 min
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 50, 125, 3, col_name = "Class1"))  # 30-60 min
classes_df.loc[classes_df["Class1"] == -1, "Class1"] = 4

# sorts the glucose values by patient id and timestamps
classes_df = classes_df.sort_values(by=["PtID", "ts"])
# reindexes based on the sorted dataset
classes_df = classes_df.reset_index(drop=True)
# nneu noch mehr neu
print(classes_df["Class1"].value_counts())

Class1
4    33113
3      973
2      431
0      388
1      231
Name: count, dtype: int64


In [10]:
import polars as pl
from typing import Sequence
from datetime import datetime




def apply_all_metrics_lazy_per_patient(
    lf: pl.LazyFrame,
    pt_col: str = "PtID",
    ts_col: str = "ts",
    column: str = "GlucoseCGM",
    sample_every: str = "5m",
    windows: Sequence[tuple[str, str]] = (("30m", "30min"), ("1h", "1hr"), ("2h", "2hr"), ("4h", "4hr"),("8h", "8hr"),("24h", "24hr")),
) -> pl.LazyFrame:
    lf = (
        lf.with_columns(pl.col(ts_col).cast(pl.Datetime))
          .sort([pt_col, ts_col])
    )

    def to_minutes(s: str) -> int:
        s = s.strip().lower()
        if s.endswith("m"):
            return int(s[:-1])
        if s.endswith("h"):
            return int(s[:-1]) * 60
        raise ValueError(f"Unsupported duration: {s} (use 'm' or 'h')")

    samp_min = to_minutes(sample_every)
    samp_dur = pl.duration(minutes=samp_min)

    for window_size, suffix in windows:
        win_min = to_minutes(window_size)
        if win_min % samp_min != 0:
            raise ValueError(f"window_size {window_size} must be a multiple of sample_every {sample_every}")

        expected_n = win_min // samp_min
        expected_span = pl.duration(minutes=samp_min * (expected_n - 1))

        # temp column names per window
        gap_col = f"__gap_{suffix}"
        span_col = f"__span_{suffix}"
        n_col = f"__n_{suffix}"
        ok_col = f"__ok_{suffix}"

        ts = pl.col(ts_col)
        c = pl.col(column).cast(pl.Float64)

        # 1) create helper columns FIRST (no nesting window->rolling)
        lf = lf.with_columns([
            ts.diff().over(pt_col).alias(gap_col),
            (ts - ts.shift(expected_n - 1)).over(pt_col).alias(span_col),

            # count non-null samples in trailing time window
            c.is_not_null()
             .cast(pl.Int32)
             .rolling_sum_by(ts, window_size=window_size, closed="right")
             .over(pt_col)
             .alias(n_col),
        ])

        # 2) compute "window complete & gap-free" mask using helper cols
        lf = lf.with_columns([
            (
                (pl.col(n_col) == expected_n) &
                (pl.col(gap_col).rolling_max(expected_n - 1).over(pt_col) <= samp_dur) &
                (pl.col(span_col) == expected_span)
            ).alias(ok_col)
        ])

        # helper: null out if window isn't ok
        def only_if_ok(expr: pl.Expr) -> pl.Expr:
            return pl.when(pl.col(ok_col)).then(expr).otherwise(None)

        # 3) add metrics (time-based rollings) masked by ok_col
        lf = lf.with_columns([
            only_if_ok(
                c.rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_mean_{suffix}"),

            only_if_ok(
                c.rolling_std_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_std_{suffix}"),

            only_if_ok(
                c.rolling_min_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_min_{suffix}"),

            only_if_ok(
                c.rolling_max_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_max_{suffix}"),

            # keep a public n_{suffix} too (null when invalid)
            only_if_ok(pl.col(n_col)).alias(f"n_{suffix}"),

            only_if_ok(
                ((c < 70).cast(pl.Int8)
                 .rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n)
                 .over(pt_col) * 100)
            ).alias(f"time_below70_{suffix}"),

            only_if_ok(
                ((c > 200).cast(pl.Int8)
                 .rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n)
                 .over(pt_col) * 100)
            ).alias(f"time_above200_{suffix}"),

            only_if_ok((c - c.shift(expected_n - 1)).over(pt_col)).alias(f"diff_{suffix}"),
            only_if_ok(((c - c.shift(expected_n - 1)).over(pt_col)) / win_min).alias(f"slope_{suffix}"),  # per minute
        ])

        # optional: drop temp cols to keep df clean
        lf = lf.drop([gap_col, span_col, n_col, ok_col])

    return lf


def add_hypo_event_window_metrics_time(
    df: pl.DataFrame,
    pt_col: str = "PtID",
    ts_col: str = "ts",
    glucose_col: str = "GlucoseCGM",
    threshold: float = 70.0,
    sample_minutes: int = 5,
    windows_min: Sequence[int] = (30, 60, 120, 240, 1440),
) -> pl.DataFrame:
    if df.is_empty():
        return df

    df = (
        df.with_columns(pl.col(ts_col).cast(pl.Datetime))
          .sort([pt_col, ts_col])
    )

    x = pl.col(glucose_col).cast(pl.Float64)
    ts = pl.col(ts_col)

    hypo = x.le(threshold).fill_null(False)    # null glucose breaks events (conservative)
    obs  = x.is_not_null()

    samp_dur = pl.duration(minutes=sample_minutes)

    # Sentinel datetime for "no event end in window"
    sentinel_ts = pl.lit(datetime(1900, 1, 1)).cast(pl.Datetime)

    out = (
        df.with_columns([
            hypo.alias("_hypo"),
            obs.alias("_obs"),
            hypo.shift(1).over(pt_col).fill_null(False).alias("_prev_hypo"),
            hypo.shift(-1).over(pt_col).fill_null(False).alias("_next_hypo"),
        ])
        .with_columns([
            (pl.col("_hypo") & ~pl.col("_prev_hypo")).alias("_event_start"),
            (pl.col("_hypo") & ~pl.col("_next_hypo")).alias("_event_end"),
        ])
        .with_columns([
            # event_id only for hypo rows; resets per patient
            pl.when(pl.col("_hypo"))
              .then(pl.col("_event_start").cast(pl.Int64).cum_sum().over(pt_col))
              .otherwise(None)
              .alias("event_id"),
        ])
        .with_columns([
            # duration only meaningful on event-end rows; else 0 (NO NULLS)
            pl.when(pl.col("_event_end"))
              .then(pl.len().over([pt_col, "event_id"]) * sample_minutes)
              .otherwise(0)
              .alias("event_duration_end_min"),

            # event-end timestamp: sentinel on non-end rows (NO NULLS)
            pl.when(pl.col("_event_end"))
              .then(ts)
              .otherwise(sentinel_ts)
              .alias("event_end_ts_filled"),

            # gap helper (per patient)
            ts.diff().over(pt_col).alias("_gap"),
        ])
    )

    exprs: list[pl.Expr] = []

    for w_min in windows_min:
        suf = "24h" if w_min == 1440 else (f"{w_min}m" if w_min < 60 else f"{w_min//60}h")
        window_size = f"{w_min}m" if w_min < 60 else f"{w_min//60}h"

        expected_n = max(1, w_min // sample_minutes)
        expected_span = pl.duration(minutes=sample_minutes * (expected_n - 1))

        # Valid-window checks (subject-wise via .over(pt_col))
        n_obs = (
            pl.col("_obs").cast(pl.Int32)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        span_ok = (ts - ts.shift(expected_n - 1)).over(pt_col) == expected_span
        max_gap_ok = (
            pl.col("_gap")
              .rolling_max(expected_n - 1)
              .over(pt_col)
              <= samp_dur
        )
        window_ok = (n_obs == expected_n) & span_ok & max_gap_ok

        def only_if_ok(e: pl.Expr) -> pl.Expr:
            return pl.when(window_ok).then(e).otherwise(None)

        # 1) hypo points in time window
        hypo_points = (
            pl.col("_hypo").cast(pl.Int32)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        exprs.append(only_if_ok(hypo_points).alias(f"hypo_points_{suf}"))

        # 2) avg event duration among events that END in the window
        # Avoid rolling_mean_by over nulls -> use sum/count (both non-null)
        dur_sum = (
            pl.col("event_duration_end_min").cast(pl.Float64)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        end_count = (
            pl.col("_event_end").cast(pl.Int32)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        avg_dur = pl.when(end_count > 0).then(dur_sum / end_count).otherwise(0.0)
        exprs.append(only_if_ok(avg_dur).fill_null(0).alias(f"avg_event_duration_min_{suf}"))

        # 3) minutes since last event END within window
        last_end = (
            pl.col("event_end_ts_filled")
              .rolling_max_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        last_end_or_null = pl.when(last_end == sentinel_ts).then(None).otherwise(last_end)
        mins_since = (ts - last_end_or_null).dt.total_minutes()
        exprs.append(only_if_ok(mins_since).fill_null(0).alias(f"min_since_last_event_{suf}"))

    return (
        out.with_columns(exprs)
           .drop([
               "_obs", "_hypo", "_prev_hypo", "_next_hypo",
               "_event_start", "_event_end", "event_id",
               "event_duration_end_min", "event_end_ts_filled", "_gap"
           ])
    )

df_polars = pl.from_pandas(classes_df)
df_polars = df_polars.sort(["PtID", "ts"])
df_polars = df_polars.unique(subset=["PtID", "ts"], keep="first")
df_features = (
    df_polars.group_by("PtID", maintain_order=True)
    .map_groups(data_preprocessing.add_time_features)) 

df_features = df_features.drop_nulls(["GlucoseCGM"])

lf_out = apply_all_metrics_lazy_per_patient(df_features.lazy())
df_out = lf_out.collect()

df_clean_out = df_out.drop_nulls(["GlucoseCGM_mean_30min", "GlucoseCGM_mean_1hr",
        "GlucoseCGM_mean_2hr"])
del df_out
del df_polars


df_in = df_features.select(["ts", "PtID", "GlucoseCGM"])

df_hypo = add_hypo_event_window_metrics_time(
    df_in,
    pt_col="PtID",
    ts_col="ts",
    glucose_col="GlucoseCGM",
    threshold=70,
    sample_minutes=5,
    windows_min=(30, 60, 120, 240, 1440),
)
del df_features
del df_in

df_hypo = df_hypo.drop("GlucoseCGM")


df_clean_out = df_clean_out.join(df_hypo, on=["PtID", "ts"], how="left")
del df_hypo


df_clean_out =df_clean_out.with_columns([
    pl.col("ts").dt.hour().alias("hour"),
    pl.col("ts").dt.minute().alias("minute"),
    pl.col("ts").dt.weekday().alias("dayofweek")
])

windows = ["30min", "1hr", "2hr", "4hr", "8hr", "24hr"]

df_clean_out = df_clean_out.with_columns([
    pl.col(f"n_{w}").is_not_null().cast(pl.Int8).alias(f"valid_{w}")
    for w in windows
])


df_imp = df_clean_out.with_columns(
    pl.selectors.numeric().fill_null(0)
)

del df_clean_out

In [11]:
df_imp


ts,GlucoseCGM,PtID,Class1,diff_5,diff_10,diff_20,diff_40,GlucoseCGM_mean_30min,GlucoseCGM_std_30min,GlucoseCGM_min_30min,GlucoseCGM_max_30min,n_30min,time_below70_30min,time_above200_30min,diff_30min,slope_30min,GlucoseCGM_mean_1hr,GlucoseCGM_std_1hr,GlucoseCGM_min_1hr,GlucoseCGM_max_1hr,n_1hr,time_below70_1hr,time_above200_1hr,diff_1hr,slope_1hr,GlucoseCGM_mean_2hr,GlucoseCGM_std_2hr,GlucoseCGM_min_2hr,GlucoseCGM_max_2hr,n_2hr,time_below70_2hr,time_above200_2hr,diff_2hr,slope_2hr,GlucoseCGM_mean_4hr,GlucoseCGM_std_4hr,…,time_below70_8hr,time_above200_8hr,diff_8hr,slope_8hr,GlucoseCGM_mean_24hr,GlucoseCGM_std_24hr,GlucoseCGM_min_24hr,GlucoseCGM_max_24hr,n_24hr,time_below70_24hr,time_above200_24hr,diff_24hr,slope_24hr,hypo_points_30m,avg_event_duration_min_30m,min_since_last_event_30m,hypo_points_1h,avg_event_duration_min_1h,min_since_last_event_1h,hypo_points_2h,avg_event_duration_min_2h,min_since_last_event_2h,hypo_points_4h,avg_event_duration_min_4h,min_since_last_event_4h,hypo_points_24h,avg_event_duration_min_24h,min_since_last_event_24h,hour,minute,dayofweek,valid_30min,valid_1hr,valid_2hr,valid_4hr,valid_8hr,valid_24hr
datetime[μs],f64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i8,i8,i8,i8,i8,i8,i8,i8,i8
2024-02-07 22:40:00,231.0,"""10_AZT125""",4,-3.0,-1.0,22.0,36.0,223.166667,10.83359,209.0,234.0,6,0.0,100.0,19.0,0.633333,211.083333,16.843981,181.0,234.0,12,0.0,66.666667,50.0,0.833333,183.0,32.154518,143.0,234.0,24,0.0,33.333333,54.0,0.45,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,40,3,1,1,1,0,0,0
2024-02-07 22:45:00,237.0,"""10_AZT125""",4,6.0,3.0,16.0,30.0,227.333333,10.48173,209.0,237.0,6,0.0,100.0,28.0,0.933333,215.75,15.451537,195.0,237.0,12,0.0,75.0,42.0,0.7,185.5,33.950091,143.0,237.0,24,0.0,37.5,60.0,0.5,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,45,3,1,1,1,0,0,0
2024-02-07 22:50:00,244.0,"""10_AZT125""",4,7.0,13.0,12.0,26.0,233.166667,7.574079,221.0,244.0,6,0.0,100.0,23.0,0.766667,219.833333,15.936427,195.0,244.0,12,0.0,83.333333,46.0,0.766667,188.291667,35.918361,143.0,244.0,24,0.0,41.666667,80.0,0.666667,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,50,3,1,1,1,0,0,0
2024-02-07 22:55:00,247.0,"""10_AZT125""",4,3.0,10.0,13.0,35.0,237.5,6.595453,231.0,247.0,6,0.0,100.0,15.0,0.5,223.916667,16.110179,195.0,247.0,12,0.0,91.666667,52.0,0.866667,191.75,37.441258,143.0,247.0,24,0.0,45.833333,95.0,0.791667,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,55,3,1,1,1,0,0,0
2024-02-07 23:00:00,247.0,"""10_AZT125""",4,0.0,3.0,16.0,38.0,240.0,6.928203,231.0,247.0,6,0.0,100.0,13.0,0.433333,228.25,14.542258,207.0,247.0,12,0.0,100.0,40.0,0.666667,195.708333,38.072561,143.0,247.0,24,0.0,50.0,99.0,0.825,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,23,0,3,1,1,1,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-02-29 23:40:00,139.0,"""9_AZT125""",4,3.0,4.0,2.0,22.0,136.5,2.345208,133.0,139.0,6,0.0,0.0,0.0,0.0,129.916667,8.80556,116.0,139.0,12,0.0,0.0,20.0,0.333333,126.166667,7.636677,116.0,139.0,24,0.0,0.0,17.0,0.141667,127.375,10.052257,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,23,40,4,1,1,1,1,0,0
2024-02-29 23:45:00,138.0,"""9_AZT125""",4,-1.0,2.0,5.0,9.0,136.333333,2.160247,133.0,139.0,6,0.0,0.0,1.0,0.033333,131.5,8.361166,116.0,139.0,12,0.0,0.0,15.0,0.25,126.833333,7.949114,116.0,139.0,24,0.0,0.0,13.0,0.108333,128.125,9.44328,…,0.0,0.0,-57.0,-0.11875,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,

In [12]:
def normalize_data(df: pl.DataFrame, value: str, new_col: str) -> pl.DataFrame:
    max_CGM = 500.0
    min_CGM = 40.0
    denom = max_CGM - min_CGM

    return df.with_columns(
        ((pl.col(value) - min_CGM) / denom).alias(new_col)
    )

In [13]:
df_imp = normalize_data(df_imp, value="GlucoseCGM", new_col="GlucoseCGM_norm")

In [14]:
df_imp

ts,GlucoseCGM,PtID,Class1,diff_5,diff_10,diff_20,diff_40,GlucoseCGM_mean_30min,GlucoseCGM_std_30min,GlucoseCGM_min_30min,GlucoseCGM_max_30min,n_30min,time_below70_30min,time_above200_30min,diff_30min,slope_30min,GlucoseCGM_mean_1hr,GlucoseCGM_std_1hr,GlucoseCGM_min_1hr,GlucoseCGM_max_1hr,n_1hr,time_below70_1hr,time_above200_1hr,diff_1hr,slope_1hr,GlucoseCGM_mean_2hr,GlucoseCGM_std_2hr,GlucoseCGM_min_2hr,GlucoseCGM_max_2hr,n_2hr,time_below70_2hr,time_above200_2hr,diff_2hr,slope_2hr,GlucoseCGM_mean_4hr,GlucoseCGM_std_4hr,…,time_above200_8hr,diff_8hr,slope_8hr,GlucoseCGM_mean_24hr,GlucoseCGM_std_24hr,GlucoseCGM_min_24hr,GlucoseCGM_max_24hr,n_24hr,time_below70_24hr,time_above200_24hr,diff_24hr,slope_24hr,hypo_points_30m,avg_event_duration_min_30m,min_since_last_event_30m,hypo_points_1h,avg_event_duration_min_1h,min_since_last_event_1h,hypo_points_2h,avg_event_duration_min_2h,min_since_last_event_2h,hypo_points_4h,avg_event_duration_min_4h,min_since_last_event_4h,hypo_points_24h,avg_event_duration_min_24h,min_since_last_event_24h,hour,minute,dayofweek,valid_30min,valid_1hr,valid_2hr,valid_4hr,valid_8hr,valid_24hr,GlucoseCGM_norm
datetime[μs],f64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i8,i8,i8,i8,i8,i8,i8,i8,i8,f64
2024-02-07 22:40:00,231.0,"""10_AZT125""",4,-3.0,-1.0,22.0,36.0,223.166667,10.83359,209.0,234.0,6,0.0,100.0,19.0,0.633333,211.083333,16.843981,181.0,234.0,12,0.0,66.666667,50.0,0.833333,183.0,32.154518,143.0,234.0,24,0.0,33.333333,54.0,0.45,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,40,3,1,1,1,0,0,0,0.415217
2024-02-07 22:45:00,237.0,"""10_AZT125""",4,6.0,3.0,16.0,30.0,227.333333,10.48173,209.0,237.0,6,0.0,100.0,28.0,0.933333,215.75,15.451537,195.0,237.0,12,0.0,75.0,42.0,0.7,185.5,33.950091,143.0,237.0,24,0.0,37.5,60.0,0.5,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,45,3,1,1,1,0,0,0,0.428261
2024-02-07 22:50:00,244.0,"""10_AZT125""",4,7.0,13.0,12.0,26.0,233.166667,7.574079,221.0,244.0,6,0.0,100.0,23.0,0.766667,219.833333,15.936427,195.0,244.0,12,0.0,83.333333,46.0,0.766667,188.291667,35.918361,143.0,244.0,24,0.0,41.666667,80.0,0.666667,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,50,3,1,1,1,0,0,0,0.443478
2024-02-07 22:55:00,247.0,"""10_AZT125""",4,3.0,10.0,13.0,35.0,237.5,6.595453,231.0,247.0,6,0.0,100.0,15.0,0.5,223.916667,16.110179,195.0,247.0,12,0.0,91.666667,52.0,0.866667,191.75,37.441258,143.0,247.0,24,0.0,45.833333,95.0,0.791667,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,22,55,3,1,1,1,0,0,0,0.45
2024-02-07 23:00:00,247.0,"""10_AZT125""",4,0.0,3.0,16.0,38.0,240.0,6.928203,231.0,247.0,6,0.0,100.0,13.0,0.433333,228.25,14.542258,207.0,247.0,12,0.0,100.0,40.0,0.666667,195.708333,38.072561,143.0,247.0,24,0.0,50.0,99.0,0.825,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,23,0,3,1,1,1,0,0,0,0.45
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-02-29 23:40:00,139.0,"""9_AZT125""",4,3.0,4.0,2.0,22.0,136.5,2.345208,133.0,139.0,6,0.0,0.0,0.0,0.0,129.916667,8.80556,116.0,139.0,12,0.0,0.0,20.0,0.333333,126.166667,7.636677,116.0,139.0,24,0.0,0.0,17.0,0.141667,127.375,10.052257,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,23,40,4,1,1,1,1,0,0,0.215217
2024-02-29 23:45:00,138.0,"""9_AZT125""",4,-1.0,2.0,5.0,9.0,136.333333,2.160247,133.0,139.0,6,0.0,0.0,1.0,0.033333,131.5,8.361166,116.0,139.0,12,0.0,0.0,15.0,0.25,126.833333,7.949114,116.0,139.0,24,0.0,0.0,13.0,0.108333,128.125,9.44328,…,0.0,-57.0,-0.11875,0.0,0.0,0.0,0

In [15]:
# Extract min and max from the scaler
x_min = df_imp["GlucoseCGM"].min()
x_max = df_imp["GlucoseCGM"].max()

# Compute scaled threshold
t_scaled = (70 - 40) / (500 - 40)
print("Scaled threshold:", t_scaled)

Scaled threshold: 0.06521739130434782


In [16]:
df_feat = df_imp.drop("Class1")
df_feat = df_feat.drop("GlucoseCGM_norm")

df_imp = df_imp.select(['ts',
 'GlucoseCGM',
 'PtID',
 'Class1',
 'GlucoseCGM_norm'])

In [17]:
# generates time series with a sliding window appraoch of 2 hour lengths for the maindatabase
# splits data into train, validation, and test
def extract_valid_windows_GLC(
    df_org: pd.DataFrame,
    timestamp_col: str = "ts",
    feature_col: str = "GlucoseCGM_norm",
    class_col: str = "Class",
    expected_sample_count: int = 25,
    min_window_duration=np.timedelta64(2, "h"),
    expected_step=np.timedelta64(5, "m"),  # <-- new: required sampling period
):
    """
    This function is used genearte time series data and splits the data into train, validation, and test data.
    Parameters:
    - df_org is the original pandas dataframe,
    - timestamp_col is the name of the column with the timestamps,
    - feature_col is the name of the column which should be returned as a time series,
    - class_col is name of the column with the classes,
    - expected_sample_count is the number of allowed continuous values in a time series,
    - min_window_duration is the allowed continuous time of a time series
    - expected_step is the expected fixed time delta between consecutive samples (e.g., 5 minutes)
    Output: It returns a list including two separate lists for X_train and Y_train
    """
    # X_train, and Y_train are initialized as empty arrays
    X_train = []
    Y_train = []
   # X_train_feat = []
    valid_ts = []
    all_ts = []


    
    # converts pandas DataFrame to polars for faster execution
    #df = pl.from_pandas(df_org)
    df =  df_org
    # sorts by timestamp to ensure chronological order
    df = df.sort(timestamp_col)
   # features = df.select(pl.exclude(["Class0","Class1", "GlucoseCGM_norm", "PtID", "ts"])).columns
    
    # extracts timestamps and total row count
    timestamps = df[timestamp_col].to_numpy()
    n_rows = df.height

    # initializes empty lists for feature windows and labels
    windows = []
    labels = []
   # windows_feature = []

    # sets starting index for window generation
    start_idx = 0

    # iterates over all rows to construct valid windows
    for end_idx in range(n_rows):
        # moves start index if window exceeds allowed duration
        while timestamps[end_idx] - timestamps[start_idx] > min_window_duration:
            start_idx += 1

        # calculates number of samples in current window
        count = end_idx - start_idx + 1

        # checks if window meets required duration and sample count
        if (
            timestamps[end_idx] - timestamps[start_idx] >= min_window_duration
            and count == expected_sample_count
        ):
            # extracts current time window
            window_df = df.slice(start_idx, count)

            # --- NEW: reject windows with gaps / irregular sampling ---
            w_ts = window_df[timestamp_col].to_numpy()
            # must be exactly expected_step between every consecutive timestamp
            if w_ts.size >= 2:
                diffs = np.diff(w_ts)
                if not np.all(diffs == expected_step):
                    continue
            # ---------------------------------------------------------

            # skips window if missing values are present
            nulls = window_df.null_count().row(0)
            if all(c == 0 for c in nulls):
                # retrieves label from last row in window
                label = window_df[class_col].to_list()[-1]
                # allows only classes 0–4 (your code actually allows 0–3)
                #if label in {0, 1, 2, 3}:
                # stores feature sequence and label
                windows.append(window_df[feature_col].to_list())
                labels.append(label)

                # feat_ids = window_df.select(features).row(-1)   
                # feat_ids = [int(v) for v in feat_ids]
                # windows_feature.append(feat_ids)
                feat_ids = window_df.select("ts").row(-1)   
                #feat_ids = [int(v) for v in feat_ids]
                valid_ts.append(feat_ids)

    # checks if any valid windows were found
    if len(windows) > 0:
        # converts to numpy arrays and reshapes data for model input
        X_data = np.array(windows).reshape(-1, expected_sample_count, 1)
        Y_data = np.array(labels).reshape(-1, 1)
       # X_data_feat = np.array(windows_feature) 

        # appends subject data to training sets
        X_train.append(X_data)
        Y_train.append(Y_data)

        X_data_feat = np.array(valid_ts) 

        all_ts.append(X_data_feat)
       # X_train_feat.append(X_data_feat)
        
        # returns feature and label arrays
        return  X_train, Y_train, all_ts #X_train, Y_train, X_train_feat
    else:
        # returns nothing if no valid window exists
        print("No valid samples generated for this subject!")
        return None


def create_X_Y(df: pl.DataFrame, sample_count=25, hours=2, modus="h", class_c="Class"):
    # split into per-subject DataFrames
    df = df.sort(["PtID", "ts"])
    groups = df.partition_by("PtID", as_dict=True)

    del df

    dic_result = {}
    for ptid, df_person in groups.items():
        out = extract_valid_windows_GLC(df_person,class_col=class_c,
            expected_sample_count=sample_count,
            min_window_duration=np.timedelta64(hours, modus))
        if out is None:
            print(f"WARNING: PtID {ptid} returned None")
            continue  

        X_data, Y_data, X_ts = out
        dic_result[ptid] = {"X": X_data, "Y": Y_data, "X_ts": X_ts}

    return dic_result

import pickle, gzip

# calls the function for time series generation with a 120 min input length 
# X_data, Y_data, X_data_feat = create_X_Y(df_imp, 25, 2, 'h', class_c = "Class1")
dic_data = create_X_Y(df_imp, 25, 2, 'h', class_c = "Class1")

In [18]:
def dict_to_allowed_pairs(d, subject_col="subject_id", ts_col="timestamp"):
    subjects = []
    timestamps = []

    for subj_key, payload in d.items():
        # subj_key is like ('001_D1NAMO',) -> get the string
        subj = subj_key[0] if isinstance(subj_key, tuple) else subj_key

        # payload["ts"] is a list of numpy arrays shaped like (n,1) with datetimes
        for arr in payload.get("X_ts", []):
            # flatten to 1D python list of datetime objects
            ts_list = np.asarray(arr).ravel().tolist()

            subjects.extend([subj] * len(ts_list))
            timestamps.extend(ts_list)

    return pl.DataFrame({subject_col: subjects, ts_col: timestamps})

keep_df = dict_to_allowed_pairs(dic_data, subject_col="PtID", ts_col="X_ts")
keep_df




PtID,X_ts
str,datetime[μs]
"""10_AZT125""",2024-02-08 05:55:00
"""10_AZT125""",2024-02-08 06:00:00
"""10_AZT125""",2024-02-08 06:05:00
"""10_AZT125""",2024-02-08 06:10:00
"""10_AZT125""",2024-02-08 06:15:00
…,…
"""9_AZT125""",2024-02-29 23:40:00
"""9_AZT125""",2024-02-29 23:45:00
"""9_AZT125""",2024-02-29 23:50:00


In [20]:
# If df["timestamp"] is Datetime, align keep_df to it:
keep_df = keep_df.rename({"X_ts": "ts"})
keep_df = keep_df.with_columns(
    pl.col("ts").cast(df_feat.schema["ts"])
)

df_filtered = df_feat.join(keep_df, on=["PtID", "ts"], how="semi")

In [21]:
df_filtered

ts,GlucoseCGM,PtID,diff_5,diff_10,diff_20,diff_40,GlucoseCGM_mean_30min,GlucoseCGM_std_30min,GlucoseCGM_min_30min,GlucoseCGM_max_30min,n_30min,time_below70_30min,time_above200_30min,diff_30min,slope_30min,GlucoseCGM_mean_1hr,GlucoseCGM_std_1hr,GlucoseCGM_min_1hr,GlucoseCGM_max_1hr,n_1hr,time_below70_1hr,time_above200_1hr,diff_1hr,slope_1hr,GlucoseCGM_mean_2hr,GlucoseCGM_std_2hr,GlucoseCGM_min_2hr,GlucoseCGM_max_2hr,n_2hr,time_below70_2hr,time_above200_2hr,diff_2hr,slope_2hr,GlucoseCGM_mean_4hr,GlucoseCGM_std_4hr,GlucoseCGM_min_4hr,…,time_below70_8hr,time_above200_8hr,diff_8hr,slope_8hr,GlucoseCGM_mean_24hr,GlucoseCGM_std_24hr,GlucoseCGM_min_24hr,GlucoseCGM_max_24hr,n_24hr,time_below70_24hr,time_above200_24hr,diff_24hr,slope_24hr,hypo_points_30m,avg_event_duration_min_30m,min_since_last_event_30m,hypo_points_1h,avg_event_duration_min_1h,min_since_last_event_1h,hypo_points_2h,avg_event_duration_min_2h,min_since_last_event_2h,hypo_points_4h,avg_event_duration_min_4h,min_since_last_event_4h,hypo_points_24h,avg_event_duration_min_24h,min_since_last_event_24h,hour,minute,dayofweek,valid_30min,valid_1hr,valid_2hr,valid_4hr,valid_8hr,valid_24hr
datetime[μs],f64,str,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i32,f64,i64,i8,i8,i8,i8,i8,i8,i8,i8,i8
2024-02-08 05:55:00,123.0,"""10_AZT125""",22.0,29.0,4.0,-3.0,111.833333,12.351788,94.0,124.0,6,0.0,0.0,-1.0,-0.033333,117.916667,10.569754,94.0,126.0,12,0.0,0.0,2.0,0.033333,110.333333,12.478387,93.0,126.0,24,0.0,0.0,30.0,0.25,106.0625,14.646772,89.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,5,55,4,1,1,1,1,0,0
2024-02-08 06:00:00,131.0,"""10_AZT125""",8.0,30.0,21.0,5.0,113.0,13.957077,94.0,131.0,6,0.0,0.0,12.0,0.4,118.75,11.209777,94.0,131.0,12,0.0,0.0,9.0,0.15,111.916667,12.593707,93.0,131.0,24,0.0,0.0,38.0,0.316667,106.020833,14.57116,89.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,6,0,4,1,1,1,1,0,0
2024-02-08 06:05:00,124.0,"""10_AZT125""",-7.0,1.0,30.0,-1.0,113.833333,14.524692,94.0,131.0,6,0.0,0.0,14.0,0.466667,118.916667,11.277156,94.0,131.0,12,0.0,0.0,0.0,0.0,113.208333,12.151147,94.0,131.0,24,0.0,0.0,30.0,0.25,105.854167,14.311117,89.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,6,5,4,1,1,1,1,0,0
2024-02-08 06:10:00,125.0,"""10_AZT125""",1.0,-6.0,24.0,1.0,116.333333,15.015547,94.0,131.0,6,0.0,0.0,31.0,1.033333,119.0,11.321741,94.0,131.0,12,0.0,0.0,-1.0,-0.016667,114.5,11.658175,94.0,131.0,24,0.0,0.0,27.0,0.225,105.729167,14.111621,89.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,6,10,4,1,1,1,1,0,0
2024-02-08 06:15:00,138.0,"""10_AZT125""",13.0,14.0,15.0,19.0,123.666667,12.452577,101.0,138.0,6,0.0,0.0,37.0,1.233333,120.0,12.468141,94.0,138.0,12,0.0,0.0,12.0,0.2,116.166667,12.049415,94.0,138.0,24,0.0,0.0,37.0,0.308333,105.958333,14.547001,89.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,6,15,4,1,1,1,1,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-02-29 23:40:00,139.0,"""9_AZT125""",3.0,4.0,2.0,22.0,136.5,2.345208,133.0,139.0,6,0.0,0.0,0.0,0.0,129.916667,8.80556,116.0,139.0,12,0.0,0.0,20.0,0.333333,126.166667,7.636677,116.0,139.0,24,0.0,0.0,17.0,0.141667,127.375,10.052257,95.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,23,40,4,1,1,1,1,0,0
2024-02-29 23:45:00,138.0,"""9_AZT125""",-1.0,2.0,5.0,9.0,136.333333,2.160247,133.0,139.0,6,0.0,0.0,1.0,0.033333,131.5,8.361166,116.0,139.0,12,0.0,0.0,15.0,0.25,126.833333,7.949114,116.0,139.0,24,0.0,0.0,13.0,0.108333,128.125,9.44328,95.0,…,0.0,0.0,-57.0,-0.11875,0.

In [22]:
def extract_valid_windows_GLC(df_org: pl.DataFrame, label_m:int):
    df =  df_org.sort(["PtID", "ts"])

    # X_train, X_val, X_test, Y_train, Y_val, and Y_test are initialized as empty arrays
    X_data = []

    X = df.drop(["PtID", "ts"]).to_numpy() 
    X_data.append(X)

    return X_data

# generates time series with sliding window approach
df_filtered = df_filtered.sort(["PtID", "ts"])
groups = df_filtered.partition_by("PtID", as_dict=True)
X_data_static = []
del df_filtered

for ptid, df_person in groups.items():
    out = extract_valid_windows_GLC(df_person, label_m=1)
    (
       X_data
    ) = out

    X_data_static.append(X_data)

In [23]:
X_patients = []
Y_patients = []
subjects = []

for subj_key, payload in dic_data.items():
    subj = subj_key[0] if isinstance(subj_key, tuple) else subj_key
    chunks_X = payload["X"]  
    chunks_Y = payload["Y"]  


    X_patient = np.concatenate(chunks_X, axis=0)  
    Y_patient = np.concatenate(chunks_Y, axis=0) 

    if X_patient.shape[0] != Y_patient.shape[0]:
        raise ValueError(f"{subj}: X and Y length mismatch: {X_patient.shape[0]} vs {Y_patient.shape[0]}")

    subjects.append(subj)
    X_patients.append(X_patient)
    Y_patients.append(Y_patient) 



def filter_out_class_5(X_list, Y_list, X_list_feat, drop_value=5):
    X_filt, Y_filt, X_filt_feat = [], [], []

    for X, Y, X_feat in zip(X_list, Y_list, X_list_feat):
        y = np.asarray(Y).reshape(-1)          # (n_i,)
        mask = y != drop_value                 # keep everything except 5

        X_filt.append(np.asarray(X)[mask])
        Y_filt.append(y[mask].reshape(-1, 1))  # keep original (n_i,1) style if you want
        X_filt_feat.append(np.asarray(X_feat)[mask])

    return X_filt, Y_filt, X_filt_feat
X_data_static = [arr for sublist in X_data_static for arr in sublist]
X_train_S1, Y_train_S1, X_feat_train_S1 = filter_out_class_5(X_patients, Y_patients, X_data_static, drop_value=4)

In [25]:
len(X_data_static)

13

In [137]:
def flatten(subject):
    array_data = [np.array(x) for x in subject]  
    flattened_data = np.concatenate(array_data, axis=0)
    return flattened_data

In [138]:
# ensures reproducability
seed_value= 42  
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)

# assuming X_data and Y_data are lists (or arrays) where each entry corresponds to one subject
subject_indices = list(range(len(Y_train_S1 )))


# flattens the arrays of single subjects into one array of shape (X,25,2)
X_train_fold = flatten(X_train_S1)
# flattens the arrays of single subjects into one array of shape (X,1)
Y_train_fold = flatten(Y_train_S1)


# initiliazes buffer size and batch size 
"""BUFFER_SIZE = len(X_train_fold) 
BATCH_SIZE = 64"""

BATCH_SIZE =  256 #1024 #512 #2048 #256
BUFFER_SIZE = min(len(X_train_fold), 10_000)
Y_test_rs = to_categorical(Y_train_fold)

# for class weights 
Y_train_weights = Y_train_fold.reshape(-1).astype('int32')


X_test_t = tf.convert_to_tensor(X_train_fold, dtype=tf.float32)
Y_test_t = tf.convert_to_tensor(Y_test_rs, dtype=tf.int32)

##### Data for static #####
# Flatten the nested structure — extract the actual arrays
X_test_feat_all = flatten(X_feat_train_S1)

Y_test_all =Y_train_fold.reshape(-1)

In [139]:
Y_test_all.shape

(1294,)

In [140]:
X_test_feat_all.shape

(1294, 83)

In [141]:
X_test_t.shape

TensorShape([1294, 25, 1])

In [142]:
Y_test_t.shape

TensorShape([1294, 4])

In [ ]:
# Save (single file)
dump((X_test_t, X_test_feat_all, Y_test_t, Y_test_all), "AZT1D.joblib", compress=("zlib", 1))

['AZT1D.joblib']